### Why these imports?

We import the core libraries we'll use throughout every module:

- **`json`** — our dataset (`intents.json`) is in JSON format; we need this to load it
- **`os`, `Path`** — to handle file paths and environment variables across platforms
- **`struct`** — to read the binary header of safetensors files in Module 8
- **`numpy`** — numerical operations: `argmax` on logits, simulating gradient descent, generating plot data
- **`torch`** — PyTorch is the deep learning framework under the hood; Hugging Face Transformers builds on it
- **`matplotlib`** — we'll visualize learning rate schedules, loss curves, and cross-entropy behaviour

We also check whether a **GPU (CUDA)** is available. If you are on an NVIDIA T4, the output will show 16 GB VRAM — enough for full fine-tuning of PhayaThaiBERT.

In [ ]:
import json
import os
import struct
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

### Key model properties to check before selecting

| Property | Where to find | PhayaThaiBERT |
|----------|--------------|---------------|
| `hidden_size` | `config.json` | 768 |
| `num_hidden_layers` | `config.json` | 12 |
| `vocab_size` | `config.json` | 32,000 (SentencePiece Thai) |
| `max_position_embeddings` | `config.json` | 514 |
| Total parameters | Model card | ~278M |
| Disk size (safetensors) | After download | ~1.1 GB |

Let's load the model and inspect these properties.

### Why we load `AutoConfig` first (not the full model)

Before downloading the full 1.1 GB model, we load just the **configuration** (~1 KB). This lets us inspect the architecture properties — hidden size, number of layers, vocabulary size — without waiting for the weights to download. Think of it as reading the label on a box before opening it.

`MODEL_NAME = "clicknext/phayathaibert"` is the Hugging Face Hub identifier. Every model on the Hub has a unique `org/model-name` string.

In [ ]:
!pip install transformers -q

In [ ]:
from transformers import AutoConfig

MODEL_NAME = "clicknext/phayathaibert"

config = AutoConfig.from_pretrained(MODEL_NAME)

print(f"Architecture          : {config.architectures}")
print(f"Hidden size           : {config.hidden_size}")
print(f"Num hidden layers     : {config.num_hidden_layers}")
print(f"Num attention heads   : {config.num_attention_heads}")
print(f"Vocab size            : {config.vocab_size}")
print(f"Max position embeddings: {config.max_position_embeddings}")

### `AutoModelForSequenceClassification` — what happens under the hood

When we load the model for classification:

1. The `Auto` classes inspect `config.json` and load the correct architecture (CamemBERT)
2. `ForSequenceClassification` adds a **classification head** on top: a linear layer mapping `768 → num_labels`
3. `ignore_mismatched_sizes=True` discards the pre-trained head and creates a fresh 3-class head
4. The BERT body retains all pre-trained weights; only the head starts random

```
Input tokens → [BERT Body: 12 layers × 768 hidden] → [CLS] representation → [Linear: 768 → 3] → logits
                 ↑ pre-trained (knows Thai)                                    ↑ new (random init)
```

### Why we build `label2id`, `id2label`, and `num_labels`

The model outputs **numbers** (logits), not text. We need a two-way mapping:

- **`label2id`** — converts intent strings to integers: `"GENERAL_CONTACT_AGENT" → 0`. This is used during training so the loss function can compare the model's numeric output to the correct label.
- **`id2label`** — converts integers back to strings: `0 → "GENERAL_CONTACT_AGENT"`. This is used at inference to translate the model's prediction into a human-readable intent name.
- **`num_labels`** — tells `AutoModelForSequenceClassification` how many output neurons the classification head needs (3 in our case).

We pass all three to `from_pretrained()` so they are stored inside `config.json` alongside the model weights. This way, anyone loading the saved model later automatically knows the label mapping.

**`ignore_mismatched_sizes=True`** is needed because the pre-trained PhayaThaiBERT originally had a different classification head (for its own pre-training tasks). This flag tells Hugging Face: "discard the old head and create a new one with `num_labels=3` output neurons." The new head starts with **random weights** — fine-tuning will teach it our intents.

In [ ]:
from transformers import AutoModelForSequenceClassification

label2id = {"GENERAL_CONTACT_AGENT": 0, "MOBILE_BILLING_CHECK_DUE_DATE": 1, "MOBILE_PLAN_INQUIRY": 2}
id2label = {v: k for k, v in label2id.items()}
num_labels = len(label2id)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
size_mb = total_params * 2 / 1e6  # FP16

print(f"Total parameters      : {total_params:,}")
print(f"Trainable parameters  : {trainable_params:,} (100% in full fine-tuning)")
print(f"Approx FP16 size      : {size_mb:.0f} MB")
print(f"\nClassifier head shape : {model.classifier.out_proj.weight.shape}")
print(f"  → Maps {model.classifier.out_proj.in_features} hidden dims to {num_labels} labels")

---

## Module 2: Tokenizer

### Why tokenization is the most important preprocessing step

Models process **numbers**, not text. The tokenizer converts human-readable text into numerical IDs and back.

**Critical rule:** You **MUST** use the same tokenizer that the model was pre-trained with. A mismatched tokenizer produces garbage because the model learned associations between specific IDs and meanings.

### SentencePiece / BPE — how PhayaThaiBERT tokenizes Thai

Thai has **no spaces between words** (unlike English). SentencePiece is a language-agnostic subword tokenization algorithm that:

1. Starts with individual characters
2. Iteratively merges the most frequent pairs (Byte-Pair Encoding / BPE)
3. Result: common words stay whole, rare words get split into subwords

Let's see this in action.

### Why we load the tokenizer with `AutoTokenizer.from_pretrained`

`AutoTokenizer.from_pretrained(MODEL_NAME)` downloads the **exact tokenizer** that PhayaThaiBERT was pre-trained with. This is not optional — you **must** use the matching tokenizer because:

- The model learned that token ID `5698` means a specific Thai subword. A different tokenizer would assign that ID to a different word, producing nonsense.
- The tokenizer defines the vocabulary size (32,000 for PhayaThaiBERT). The model's embedding layer has exactly 32,000 rows — one per token ID.

We then tokenize a sample Thai sentence to see how it gets split into subwords and converted to integer IDs. `.tokenize()` shows the subword pieces, `.encode()` shows the numeric IDs, and `.decode()` converts them back to text — proving the process is reversible.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

sample = "บิลเดือนนี้ตัดวันไหนครับ"

print(f"Original text  : {sample}")
print(f"Tokens         : {tokenizer.tokenize(sample)}")
print(f"Token IDs      : {tokenizer.encode(sample)}")
print(f"Decoded back   : {tokenizer.decode(tokenizer.encode(sample))}")
print(f"\nVocab size     : {tokenizer.vocab_size}")
print(f"Model max len  : {tokenizer.model_max_length}")

### The four key outputs of a tokenizer

| Output | What it is | Shape |
|--------|-----------|-------|
| `input_ids` | Token IDs (integers) | `[1, seq_len]` |
| `attention_mask` | 1 = real token, 0 = padding | `[1, seq_len]` |
| `token_type_ids` | Sentence A vs B (not used for single-input tasks) | `[1, seq_len]` |
| `special_tokens_mask` | Marks `<s>`, `</s>`, `<pad>` positions | `[1, seq_len]` |

### Why we call `tokenizer(sample, return_tensors="pt")`

When we pass `return_tensors="pt"`, the tokenizer returns **PyTorch tensors** (not plain lists). This is the format the model expects as input. We inspect the returned dictionary to see all the keys (`input_ids`, `attention_mask`) and their shapes — this helps us understand exactly what the model receives when we call `model(**encoded)`.

In [ ]:
encoded = tokenizer(sample, return_tensors="pt")

print("Keys returned:", list(encoded.keys()))
print()
for key, val in encoded.items():
    print(f"{key:20s} shape={list(val.shape)}  values={val[0].tolist()}")

### Special tokens and their roles

| Token | ID | Purpose |
|-------|----|---------|
| `<s>` | BOS/CLS | Beginning of sequence — the model aggregates meaning into this position |
| `</s>` | EOS/SEP | End of sequence — signals the input is complete |
| `<pad>` | Padding | Fills sequences to uniform length in a batch (model ignores via attention_mask) |
| `<unk>` | Unknown | Fallback for characters not in the vocabulary |
| `<mask>` | Mask | Used during MLM pre-training (not used in fine-tuning) |

### Why we print each special token and its ID

Each special token has a specific integer ID baked into the vocabulary. We print them here so you can recognise them when you see raw `input_ids` — for example, if you see `[5, ..., 6]`, you now know that `5` is `<s>` (start) and `6` is `</s>` (end), not actual Thai words.

In [ ]:
print("Special tokens:")
for name in ["bos_token", "eos_token", "pad_token", "unk_token", "mask_token"]:
    token = getattr(tokenizer, name)
    token_id = tokenizer.convert_tokens_to_ids(token)
    print(f"  {name:15s} = '{token}' (ID: {token_id})")

### Truncation, padding, and `MAX_LENGTH`

Models have a **maximum input size** (like a form with a character limit). For PhayaThaiBERT this is **514 tokens**.

- **Truncation** (`truncation=True, max_length=512`): if the text exceeds 512 tokens, cut from the right
- **Padding** (`padding=True`): if the text is shorter, fill with `<pad>` tokens so all sequences in a batch have the same length

#### Static vs dynamic padding

- **Static padding** (`padding="max_length"`): always pad to 512 — wastes compute on short inputs
- **Dynamic padding** (`DataCollatorWithPadding`): pad each batch to the longest sequence in that batch

If the longest text in a batch is 47 tokens, dynamic padding only pads to 47 (not 512). This **saves significant compute**.

In the training notebook, we use `padding=False` during tokenization and let `DataCollatorWithPadding` handle it per batch.

### Why we compare short vs long text and static vs dynamic padding

This cell demonstrates three critical tokenizer behaviours side by side:

1. **Truncation in action:** A short text stays at its natural length; a very long text gets cut to `max_length=512` tokens. Without truncation the model would crash because it cannot handle sequences longer than its `max_position_embeddings`.
2. **Static padding waste:** Padding a 5-token sentence to 512 tokens adds 507 useless `<pad>` tokens — the GPU processes all of them for nothing.
3. **Dynamic padding efficiency:** `DataCollatorWithPadding` pads only to the longest sequence in the current batch. If the longest is 12 tokens, every sequence gets padded to 12 — not 512. This is what our training notebook uses.

In [ ]:
short_text = "สวัสดีครับ"
long_text = "สวัสดีครับ " * 200

enc_short = tokenizer(short_text, truncation=True, max_length=512)
enc_long = tokenizer(long_text, truncation=True, max_length=512)

print(f"Short text tokens : {len(enc_short['input_ids'])} (no truncation needed)")
print(f"Long text tokens  : {len(enc_long['input_ids'])} (truncated to max_length)")

print("\n--- Static padding (max_length=512) ---")
enc_padded = tokenizer(short_text, truncation=True, max_length=512, padding="max_length")
print(f"Padded length: {len(enc_padded['input_ids'])} tokens")
print(f"Attention mask sum: {sum(enc_padded['attention_mask'])} real tokens, {512 - sum(enc_padded['attention_mask'])} padding")

print("\n--- Dynamic padding (per batch) ---")
from transformers import DataCollatorWithPadding

collator = DataCollatorWithPadding(tokenizer=tokenizer)
batch = collator([
    tokenizer("สวัสดีครับ", truncation=True, max_length=512),
    tokenizer("ขอเช็กแพ็กเกจปัจจุบันครับ", truncation=True, max_length=512),
])
print(f"Batch padded to: {batch['input_ids'].shape[1]} tokens (shortest needed to fit both)")

### Tokenizer files explained

When you save a tokenizer, several files are written:

| File | Content |
|------|---------|
| `tokenizer.json` | Full vocabulary + merge rules (~17 MB for PhayaThaiBERT) |
| `tokenizer_config.json` | Settings: model_max_length, special tokens, tokenizer class |
| `special_tokens_map.json` | Explicit mapping of special token names to their string values |

---

## Module 3: Fine-Tuning — Full, LoRA, QLoRA, PEFT, SFT, BitsAndBytes

This is the most conceptually dense module. We'll build understanding as a progression:

```
Full Fine-Tuning → Why it breaks for large models → PEFT (umbrella) → LoRA → QLoRA + BitsAndBytes → SFT
```

### 3.1 What is fine-tuning?

The pre-trained model knows Thai but not your task. Fine-tuning **continues training** on your labeled data so it learns to map text to your intent labels.

In **full fine-tuning**, every one of the model's parameters is updated through gradient descent.

### Why we count total vs trainable parameters

This confirms that in **full fine-tuning**, 100% of the model's parameters are trainable — every single weight will be updated during training. Later (Module 3.4), we'll see how LoRA reduces this to 0.1-1%. The print also shows the trainable percentage, which becomes a key comparison metric across fine-tuning strategies.

In [ ]:
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Full fine-tuning:")
print(f"  Total parameters    : {total:>15,}")
print(f"  Trainable parameters: {trainable:>15,}")
print(f"  Trainable %         : {trainable/total*100:.1f}%")

### 3.2 Why full fine-tuning works here but fails for large models

During training, GPU memory must hold:
1. **Model weights** (FP16)
2. **Gradients** (same size as weights)
3. **Optimizer states** (Adam stores 2 additional copies: momentum + variance)

**Rule of thumb:** full fine-tuning needs roughly **4x the model size** in GPU memory.

| Model | Parameters | FP16 Size | 4x Memory | T4 16 GB Fits? |
|-------|-----------|-----------|-----------|----------------|
| PhayaThaiBERT | 278M | ~556 MB | ~2.2 GB | **Yes, easily** |
| Qwen-2.5-7B | 7B | ~14 GB | ~56 GB | No |
| Llama-3-70B | 70B | ~140 GB | ~560 GB | Absolutely not |

This is why we need parameter-efficient alternatives for large models.

### Why we compute GPU memory for three different models

This table makes the memory problem **concrete**. Instead of abstract rules, you can see the exact numbers: PhayaThaiBERT needs 2.2 GB (fits on T4), Qwen-7B needs 56 GB (does not fit), Llama-70B needs 560 GB (needs a data centre). This is the motivation for everything in the next sections — LoRA, QLoRA, and BitsAndBytes exist because full fine-tuning simply cannot fit large models on a single GPU.

In [ ]:
models_info = [
    ("PhayaThaiBERT",  278e6),
    ("Qwen-2.5-7B",   7e9),
    ("Llama-3-70B",   70e9),
]

t4_vram_gb = 16

print(f"{'Model':<20} {'Params':>10} {'FP16 Size':>12} {'Full FT Memory':>16} {'Fits T4?':>10}")
print("-" * 72)
for name, params in models_info:
    fp16_gb = params * 2 / 1e9
    ft_gb = fp16_gb * 4
    fits = "Yes" if ft_gb < t4_vram_gb else "No"
    print(f"{name:<20} {params/1e9:>8.1f}B {fp16_gb:>10.1f} GB {ft_gb:>14.1f} GB {fits:>10}")

### 3.3 PEFT: Parameter-Efficient Fine-Tuning (the umbrella)

PEFT is a **family of techniques** that fine-tune only a small subset of parameters. Instead of updating billions of parameters, you update only 0.1-1% of them while keeping the rest frozen.

The `peft` library from Hugging Face implements several PEFT methods:

| Method | What it does | Trainable % |
|--------|-------------|-------------|
| **LoRA** | Low-rank adapter matrices | 0.1-1% |
| **QLoRA** | LoRA + 4-bit quantized base model | 0.1-1% |
| **Prefix Tuning** | Learnable "virtual tokens" prepended to input | < 0.1% |
| **IA3** | Scales activations with learned vectors | < 0.1% |
| **Adapters** | Small bottleneck layers between transformer layers | 0.5-3% |

### 3.4 LoRA (Low-Rank Adaptation) — the breakthrough

**Key insight:** During fine-tuning, the weight updates tend to be **low-rank** — they lie in a small subspace. We can capture this efficiently.

Instead of updating a full weight matrix `W` (e.g., 768 x 768 = 589,824 parameters), LoRA decomposes the update into two small matrices:

```
Original:   W (768 x 768) + delta_W (768 x 768)          = 589,824 trainable params
LoRA:       W (frozen)    + A(768 x r) x B(r x 768)      = 12,288 trainable params (when r=8)
```

**Visual representation:**

```
Input (768)                      Output (768)
     ├───── [Frozen W: 768x768] ──────┤
     │                                 │  (added together)
     └── [A: 768→8] → [B: 8→768] ─────┘
            trainable   trainable
```

#### LoRA key parameters

| Parameter | Meaning | Typical values |
|-----------|---------|----------------|
| `r` (rank) | Size of the low-rank matrices | 4, 8, 16, 32, 64 |
| `lora_alpha` | Scaling factor (controls adapter contribution) | Usually `2 x r` |
| `target_modules` | Which layers to apply LoRA to | `q_proj`, `v_proj`, `k_proj`, `o_proj` |
| `lora_dropout` | Dropout on LoRA layers | 0.0 - 0.1 |

### Why we calculate LoRA parameter savings for different ranks

This table shows the **concrete trade-off** when choosing a LoRA rank `r`. A rank of 8 reduces trainable parameters by 98% per layer. A rank of 64 still saves 83%. You pick the rank based on how much capacity the adapter needs: simple tasks (like our 3-class intent) work with low ranks; complex tasks may need higher ranks.

In [ ]:
hidden = 768

print("LoRA parameter savings per linear layer (768 x 768):")
print(f"{'Rank (r)':>10} {'Full params':>15} {'LoRA params':>15} {'Reduction':>12}")
print("-" * 55)
for r in [4, 8, 16, 32, 64]:
    full = hidden * hidden
    lora = hidden * r + r * hidden
    print(f"{r:>10} {full:>15,} {lora:>15,} {(1 - lora/full)*100:>10.1f}%")

### 3.5 QLoRA (Quantized LoRA) — LoRA on a diet

QLoRA = LoRA + **4-bit quantization** of the base model.

- The frozen base model is compressed to **4 bits per parameter** (instead of 16)
- Only the small LoRA adapter weights are kept in FP16/BF16 for training
- Computation happens in BF16 (dequantize on-the-fly → compute → discard)

| Method | 7B Model Memory |
|--------|----------------|
| Full fine-tuning FP16 | ~56 GB |
| LoRA FP16 base | ~14 GB + small adapter |
| **QLoRA 4-bit base** | **~3.5 GB + small adapter** |

QLoRA makes it possible to fine-tune a **7B model on a single T4 (16 GB)**.

### Why we compare Full FT / LoRA / QLoRA memory side by side

This is the payoff of the entire Module 3 progression. For a 7B model, QLoRA reduces total memory from **56 GB to ~3.7 GB** — a 15x reduction. The table breaks it down into base model, adapter, and optimizer memory so you can see exactly where each method saves.

In [ ]:
print("Memory comparison for a 7B parameter model:")
print(f"{'Method':<25} {'Base Model':>12} {'Adapter':>10} {'Optimizer':>12} {'Total':>10}")
print("-" * 72)

params_7b = 7e9
lora_params = params_7b * 0.005  # ~0.5% trainable

full_base = params_7b * 2 / 1e9
full_opt = full_base * 3  # gradients + 2 adam states
print(f"{'Full FT (FP16)':<25} {full_base:>10.1f}GB {'—':>10} {full_opt:>10.1f}GB {full_base + full_opt:>8.1f}GB")

lora_adapter = lora_params * 2 / 1e9
lora_opt = lora_adapter * 3
print(f"{'LoRA (FP16 base)':<25} {full_base:>10.1f}GB {lora_adapter:>8.2f}GB {lora_opt:>10.2f}GB {full_base + lora_adapter + lora_opt:>8.1f}GB")

qlora_base = params_7b * 0.5 / 1e9  # 4-bit
print(f"{'QLoRA (4-bit base)':<25} {qlora_base:>10.1f}GB {lora_adapter:>8.2f}GB {lora_opt:>10.2f}GB {qlora_base + lora_adapter + lora_opt:>8.1f}GB")

### 3.6 BitsAndBytes — the quantization engine

`bitsandbytes` is the library that implements 4-bit and 8-bit quantization. It provides `BitsAndBytesConfig` which you pass when loading a model.

```python
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",             # NormalFloat4 — optimal for normally distributed weights
    bnb_4bit_compute_dtype=torch.bfloat16,  # compute in BF16 for speed
    bnb_4bit_use_double_quant=True,         # quantize the quantization constants too
)

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-7B",
    quantization_config=bnb_config,
)
```

| Parameter | Meaning |
|-----------|---------|
| `load_in_4bit=True` | Quantize all linear layers to 4-bit |
| `bnb_4bit_quant_type="nf4"` | NormalFloat4 — a special 4-bit type designed for neural network weights (normally distributed) |
| `bnb_4bit_compute_dtype=bfloat16` | Dequantize to BF16 on-the-fly for computation |
| `bnb_4bit_use_double_quant=True` | Quantizes the scaling factors themselves — saves another ~0.4 bits/param |

> **Not used in our PhayaThaiBERT notebook** because the model is small enough for full fine-tuning. You would use this when training the **Qwen decoder** in Phase 2.

### 3.7 SFT: Supervised Fine-Tuning

SFT = fine-tuning with **labeled input-output pairs**. It is the general term for "learning from examples."

| Context | Input | Output | Library |
|---------|-------|--------|---------|
| **Encoders** (our case) | Thai chat text | Intent label | Hugging Face `Trainer` |
| **Decoders** (Qwen, Llama) | Instruction/prompt | Expected response text | `trl.SFTTrainer` |

Our training notebook performs SFT, though for encoder classification we simply call it "fine-tuning."

For decoder models, the `trl` library provides `SFTTrainer` which extends the Hugging Face `Trainer` with:
- Chat template formatting
- Packing (fit multiple short examples into one sequence to maximize throughput)
- Native integration with PEFT/LoRA configs

**SFT vs RLHF:**
- SFT: learns directly from example input-output pairs
- RLHF (Reinforcement Learning from Human Feedback): adds a reward model + human preference feedback on top of SFT
- Typical pipeline: Pre-train → SFT → RLHF (or DPO as a simpler alternative)

### 3.8 Putting it all together — the code path for each method


##### === FULL FINE-TUNING (our notebook for PhayaThaiBERT) ===
```python
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
trainer = Trainer(model=model, args=training_args, ...)
trainer.train()  # all 278M params updated
```

##### === LoRA (for medium models) ===
```python
from peft import LoraConfig, get_peft_model

model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-7B")
lora_config = LoraConfig(r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"])
model = get_peft_model(model, lora_config)  # freezes base, adds LoRA layers
trainer = Trainer(model=model, ...)
trainer.train()  # only ~0.5% params updated
```
##### === QLoRA (for large models on limited GPU) ===
```python
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", ...)
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-7B", quantization_config=bnb_config)
model = get_peft_model(model, lora_config)
trainer = Trainer(model=model, ...)
trainer.train()  # 4-bit base + FP16 LoRA adapters
```

### 3.9 Why our notebook uses full fine-tuning

| Factor | PhayaThaiBERT | Qwen-7B (Phase 2) |
|--------|--------------|-------------------|
| Parameters | 278M | 7B |
| FP16 size | 556 MB | 14 GB |
| Full FT memory | ~2.2 GB | ~56 GB |
| T4 fits? | **Yes** | No |
| Method | **Full fine-tuning** | **QLoRA (4-bit + LoRA)** |

---

## Module 4: Learning Rate

### The single most important hyperparameter

The learning rate controls **how much** the model weights change at each training step.

Formally:

```
new_weight = old_weight - learning_rate x gradient
```

### Intuition: walking downhill blindfolded

Imagine you are on a hilly landscape (the **loss surface**) trying to find the lowest valley (minimum loss).

- **Learning rate = your step size**
- **Too large** (e.g., `1e-2`): you leap over valleys, bounce around, may never converge — loss oscillates or explodes
- **Too small** (e.g., `1e-7`): you take tiny baby steps, barely move — training takes forever, may get stuck in a local minimum
- **Just right** (`2e-5` for BERT fine-tuning): steady progress downhill

### Why `2e-5` for fine-tuning BERT?

The original BERT paper (Devlin et al., 2019) tested learning rates in `{2e-5, 3e-5, 5e-5}` and found `2e-5` works well across tasks.

Fine-tuning learning rates are **much smaller** than pre-training rates because the model already knows a lot. You want to **gently adjust**, not destroy existing knowledge. Overwriting pre-trained knowledge is called **catastrophic forgetting**.

### Why we simulate gradient descent with three learning rates

Rather than just saying "too high is bad," we **show it**. This code simulates a simple optimization problem — finding the minimum of `L = (w - 3)²` — with three learning rates:

- **LR = 0.8 (too large):** the loss bounces wildly and never converges — the optimizer keeps overshooting the minimum.
- **LR = 0.05 (just right):** the loss drops quickly and stabilizes near zero.
- **LR = 0.002 (too small):** the loss barely moves after 100 steps — training would take forever.

The real model uses `2e-5`, but the principle is identical. This visual makes the concept stick.

In [ ]:
np.random.seed(42)

def simulate_training(lr, steps=100, label=""):
    """Simulate gradient descent on a simple quadratic loss: L = (w - 3)^2."""
    w = 10.0  # start far from optimum
    losses = []
    for _ in range(steps):
        loss = (w - 3) ** 2
        grad = 2 * (w - 3) + np.random.normal(0, 0.5)  # noisy gradient
        w = w - lr * grad
        losses.append(loss)
    return losses

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

configs = [
    (0.8, "LR = 0.8 (too large)", "red"),
    (0.05, "LR = 0.05 (just right)", "green"),
    (0.002, "LR = 0.002 (too small)", "blue"),
]

for ax, (lr, title, color) in zip(axes, configs):
    losses = simulate_training(lr)
    ax.plot(losses, color=color, linewidth=2)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel("Step")
    ax.set_ylabel("Loss")
    ax.set_ylim(-1, 60)
    ax.grid(True, alpha=0.3)

plt.suptitle("Effect of Learning Rate on Training", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### Learning rate schedules

The Hugging Face `Trainer` uses **linear warmup + linear decay** by default.

```
LR
2e-5 |       /‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾\
     |      /                   \
     |     /                     \
  0  |____/                       \____
     0   warmup    training       end
```

- **Warmup:** start from 0, gradually increase to the target LR over the first few steps. Why? Early gradients are noisy because the classification head is randomly initialized — small steps first, then speed up.
- **Decay:** gradually decrease LR toward 0 at the end of training — fine-grained adjustments near the end.

### Weight decay: the partner to learning rate

`weight_decay=0.01` in our notebook adds L2 regularization: at each step, weights are gently pushed toward zero.

```
new_weight = old_weight - lr * gradient - lr * weight_decay * old_weight
```

This prevents weights from growing too large, which reduces overfitting.

### Why we plot the warmup + decay schedule

The Hugging Face `Trainer` does not use a fixed learning rate — it follows a **schedule**. This plot shows exactly what happens: the LR starts at 0, warms up over the first 30 steps, then linearly decays back to 0 by the end of training. Warmup prevents wild early updates (when the random classification head produces noisy gradients), and decay enables fine-grained adjustments at the end.

In [ ]:
total_steps = 300
warmup_steps = 30
max_lr = 2e-5

schedule = []
for step in range(total_steps):
    if step < warmup_steps:
        lr = max_lr * step / warmup_steps
    else:
        lr = max_lr * (1 - (step - warmup_steps) / (total_steps - warmup_steps))
    schedule.append(lr)

plt.figure(figsize=(10, 4))
plt.plot(schedule, color="teal", linewidth=2)
plt.axvline(x=warmup_steps, color="gray", linestyle="--", alpha=0.5, label="End of warmup")
plt.xlabel("Training Step")
plt.ylabel("Learning Rate")
plt.title("Linear Warmup + Linear Decay Schedule")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## Module 5: Loss Function

### What is a loss function?

The loss function is the **score** that tells the model how wrong its predictions are. The model's entire training goal is to **minimize the loss**.

- Lower loss = better predictions
- The loss must be **differentiable** so we can compute gradients (the direction to improve)

### Cross-Entropy Loss

`AutoModelForSequenceClassification` automatically uses **Cross-Entropy Loss** — you never see it explicitly in the training notebook; it is built into the model's `forward()` method.

**Step-by-step example:**

1. Model outputs raw logits: `[-0.088, 0.399, -0.763]` (one score per class)
2. Softmax converts to probabilities: `[0.319, 0.519, 0.162]` (sum to 1.0)
3. True label is class 1 (BILLING), so the "correct" probability is `0.519`
4. Cross-entropy = `-log(0.519)` = `0.656`
5. Perfect prediction (prob=1.0) → loss = `-log(1.0)` = **0**
6. Terrible prediction (prob=0.01) → loss = `-log(0.01)` = **4.6**

### Why cross-entropy and not something simpler?

| Loss | Formula | Problem |
|------|---------|--------|
| Accuracy | correct / total | Not differentiable — cannot compute gradients |
| Mean Squared Error | (predicted - true)^2 | Works but poorly calibrated for classification |
| **Cross-Entropy** | -log(P(correct class)) | Differentiable, strongly penalizes confident wrong answers |

### Why we walk through cross-entropy with real numbers

Instead of just showing the formula, we **compute every step** using the actual logits from our model. You can trace: raw logits → softmax probabilities → pick the correct class → compute `-log(probability)`. The "extremes" table below shows why cross-entropy is so effective: a highly confident wrong answer (P=0.01) gets a loss of 4.6, while a perfect prediction gets 0. This strong penalty forces the model to avoid overconfident mistakes.

In [ ]:
logits = torch.tensor([-0.088, 0.399, -0.763])
true_label = 1  # MOBILE_BILLING_CHECK_DUE_DATE

probs = torch.softmax(logits, dim=-1)
ce_loss = -torch.log(probs[true_label])

print("=== Cross-Entropy Step by Step ===")
print(f"Raw logits          : {logits.tolist()}")
print(f"After softmax (prob): {[round(p, 4) for p in probs.tolist()]}")
print(f"True label          : {true_label} ({id2label[true_label]})")
print(f"P(correct class)    : {probs[true_label]:.4f}")
print(f"CE loss = -log({probs[true_label]:.4f}) = {ce_loss:.4f}")

print("\n=== What happens at extremes ===")
for p in [1.0, 0.9, 0.5, 0.1, 0.01]:
    loss = -np.log(p)
    print(f"  P(correct) = {p:.2f} → loss = {loss:.4f}")

### Why we plot the cross-entropy curve

This visualization makes the key property of cross-entropy immediately obvious: the curve **explodes** as the probability of the correct class approaches zero. A model that confidently predicts the wrong class gets punished with a very high loss — much more than a model that is merely unsure. This asymmetric penalty is why cross-entropy is the standard loss for classification.

In [ ]:
probs_range = np.linspace(0.01, 1.0, 200)
ce_values = -np.log(probs_range)

plt.figure(figsize=(8, 5))
plt.plot(probs_range, ce_values, color="crimson", linewidth=2)
plt.xlabel("P(correct class)", fontsize=12)
plt.ylabel("Cross-Entropy Loss", fontsize=12)
plt.title("Cross-Entropy Loss: strongly penalizes confident wrong answers", fontsize=13)

plt.annotate("Perfect prediction\nloss = 0", xy=(0.98, 0.02), fontsize=10,
             ha="right", color="green")
plt.annotate("Confident & wrong\nloss explodes!", xy=(0.05, 3.0), fontsize=10,
             color="red")

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Watching loss during training

- **Training loss:** measured on training data — should decrease steadily
- **Validation loss:** measured on held-out data — should decrease then plateau
- If training loss keeps dropping but validation loss **rises** → **overfitting** (model memorizes training data instead of learning patterns)

```
Loss
2.0 |\.               ╱ validation rising = OVERFITTING
    | \.___         ╱
1.0 |  \   ‾‾‾---‾
    |   \          validation (should plateau)
0.5 |    \____
    |         ‾‾‾‾‾  training (keeps dropping)
    └─────────────────
    Epoch 1   2   3
```

In our notebook, `logging_steps=25` logs the training loss every 25 optimizer steps.

### Why we plot healthy training vs overfitting

These two plots side by side teach you **what to watch for** during a real training run:

- **Left (healthy):** both training and validation loss decrease — the model is learning generalisable patterns.
- **Right (overfitting):** training loss keeps dropping but validation loss starts **rising** around step 60 — the model is memorizing training examples instead of learning patterns. This is when you should stop training (or add regularisation like weight decay, dropout, or fewer epochs).

In our training notebook, `load_best_model_at_end=True` handles this automatically by reverting to the checkpoint with the best F1.

In [ ]:
np.random.seed(42)
steps = np.arange(1, 101)

train_loss = 2.0 * np.exp(-0.04 * steps) + 0.1 + np.random.normal(0, 0.03, len(steps))

val_good = 2.0 * np.exp(-0.035 * steps) + 0.25 + np.random.normal(0, 0.04, len(steps))
val_overfit = val_good.copy()
val_overfit[60:] = val_good[60] + 0.008 * (steps[60:] - steps[60]) + np.random.normal(0, 0.03, len(steps[60:]))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(steps, train_loss, label="Training loss", color="blue", alpha=0.7)
ax1.plot(steps, val_good, label="Validation loss", color="orange", alpha=0.7)
ax1.set_title("Healthy training: both losses decrease", fontsize=13)
ax1.set_xlabel("Step")
ax1.set_ylabel("Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(steps, train_loss, label="Training loss", color="blue", alpha=0.7)
ax2.plot(steps, val_overfit, label="Validation loss", color="red", alpha=0.7)
ax2.axvline(x=60, color="gray", linestyle="--", alpha=0.5)
ax2.annotate("Overfitting starts here!", xy=(62, val_overfit[60] + 0.1),
             fontsize=11, color="red")
ax2.set_title("Overfitting: validation loss rises", fontsize=13)
ax2.set_xlabel("Step")
ax2.set_ylabel("Loss")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## Module 6: F1 Score and Metrics

### From logits to prediction

```
logits = [-0.088, 0.399, -0.763]   ← raw model output (one per class)
pred   = argmax(logits) = 1         ← index of highest value
label  = id2label[1] = "MOBILE_BILLING_CHECK_DUE_DATE"
```

### Accuracy — simple but misleading

Accuracy = correct predictions / total predictions.

**Problem:** If 90% of data is one class, predicting that class for everything gives 90% accuracy — but the model learned nothing useful. We need better metrics.

### Precision, Recall, F1

Think of searching for all BILLING examples:

- **Precision:** "Of all examples the model *labeled* as BILLING, what fraction were actually BILLING?" → measures false positives
- **Recall:** "Of all *actual* BILLING examples, what fraction did the model catch?" → measures false negatives
- **F1 Score:** Harmonic mean of precision and recall — balances both

$$F1 = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$

Why **harmonic** mean instead of simple average?
- Simple average: precision=1.0, recall=0.01 → average=0.505 (looks decent!)
- Harmonic mean: precision=1.0, recall=0.01 → F1=**0.02** (correctly shows the model is broken)

### Macro vs Weighted F1

- **Macro F1:** Average F1 across all classes **equally** → treats rare classes the same as common ones
- **Weighted F1:** Average weighted by number of examples per class → common classes dominate

Our notebook uses **macro** F1 because all intents should matter equally regardless of frequency.

### Why we build a confusion matrix and classification report from scratch

Rather than waiting for a full training run, we use **simulated predictions** (`y_true` vs `y_pred`) to show exactly what a confusion matrix and classification report look like. This way you can practice reading the report — which columns matter, what "support" means, where to spot weak classes — before encountering it in the real training notebook.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_true = [0, 0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2]
y_pred = [0, 0, 0, 0, 1, 1, 1, 1, 2, 0, 2, 2]

labels = ["CONTACT_AGENT", "BILLING", "PLAN_INQUIRY"]

print("=== Confusion Matrix ===")
print("(Rows = Actual, Columns = Predicted)\n")
cm = confusion_matrix(y_true, y_pred)
header = f"{'':>18}" + "".join(f"{l[:8]:>10}" for l in labels)
print(header)
for i, row in enumerate(cm):
    print(f"  Actual {labels[i][:8]:<10}" + "".join(f"{v:>10}" for v in row))

print("\n=== Classification Report ===")
print(classification_report(y_true, y_pred, target_names=labels, zero_division=0))

### Reading the classification report

| Column | Meaning |
|--------|---------|
| `precision` | Of all predicted as this class, how many were correct? |
| `recall` | Of all actual examples of this class, how many did we catch? |
| `f1-score` | Harmonic mean of precision and recall |
| `support` | Number of actual examples for this class in the test set |
| `macro avg` | Unweighted average across all classes |
| `weighted avg` | Weighted by support |

### Why `metric_for_best_model="f1"` in our notebook

The Trainer saves a checkpoint after each epoch. At the end, it loads the checkpoint with the **highest F1 score** (not the lowest loss). Why? F1 directly measures classification quality, while loss can improve numerically even when predictions get worse on edge cases.

### Why we compare harmonic mean vs simple average

This table proves **why F1 uses the harmonic mean** instead of a simple average. Look at the first row: precision=1.0 and recall=0.01 (the model catches almost nothing). A simple average says 0.505 — that looks okay! But the harmonic mean (F1) says **0.02** — correctly flagging that the model is broken. The harmonic mean forces **both** precision and recall to be high for a good F1 score.

In [ ]:
print("=== Why harmonic mean matters ===")
print(f"{'Precision':>10} {'Recall':>10} {'Simple Avg':>12} {'F1 (harmonic)':>15}  {'Verdict':>10}")
print("-" * 62)
cases = [(1.0, 0.01), (0.9, 0.9), (0.5, 0.5), (1.0, 1.0), (0.7, 0.3)]
for p, r in cases:
    avg = (p + r) / 2
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
    verdict = "Good" if f1 > 0.6 else "Bad"
    print(f"{p:>10.2f} {r:>10.2f} {avg:>12.3f} {f1:>15.3f}  {verdict:>10}")

---

## Module 7: Epochs and Other Hyperparameters

### Epochs

**1 epoch** = the model sees every training example exactly once.

`num_train_epochs=3` means the model reads the entire dataset **3 times**.

- **More epochs:** model learns more patterns, but risks **overfitting** (memorizing training data)
- **Fewer epochs:** model may not learn enough (**underfitting**)
- **Rule of thumb for BERT fine-tuning:** 2-5 epochs (the BERT paper recommends 2-4)

### Batch size

`per_device_train_batch_size=8`

Number of examples processed before one weight update. Affects:

| Aspect | Larger batch | Smaller batch |
|--------|-------------|---------------|
| GPU memory | More | Less |
| Gradient quality | Smoother (less noisy) | Noisier (can help generalize) |
| Steps per epoch | Fewer | More |
| Wall-clock speed | Faster (fewer updates) | Slower (more updates) |

### Gradient accumulation

`gradient_accumulation_steps=1`

Simulates a larger batch by accumulating gradients across multiple forward passes before updating weights.

- `batch_size=8` with `accumulation=4` → **effective batch size of 32**
- Useful when you want large batches but GPU cannot fit them in one pass
- Set to 1 in our notebook (small model, not needed)

### Weight decay

`weight_decay=0.01`

L2 regularization: at each step, weights are gently pushed toward zero to prevent them from growing too large.

### FP16 (mixed precision training)

`fp16=torch.cuda.is_available()`

Use 16-bit floating point instead of 32-bit.

| Aspect | FP32 | FP16 |
|--------|------|------|
| Bits per number | 32 | 16 |
| Memory per param | 4 bytes | 2 bytes |
| Speed | Baseline | ~2x faster |
| Precision | Higher | Slightly lower (acceptable) |

The Trainer handles the complexity: forward pass in FP16 for speed, loss scaling to prevent underflow, master weights in FP32 for accuracy.

> **BF16 (bfloat16):** An alternative with a wider exponent range. Preferred on Ampere+ GPUs (A100), but T4 only supports FP16.

### Checkpointing strategy

```python
save_strategy="epoch"           # save after each epoch
save_total_limit=2              # keep only 2 most recent (save disk)
load_best_model_at_end=True     # after training, revert to best checkpoint (by F1)
metric_for_best_model="f1"      # use F1 to determine "best"
```

### Why we create `TrainingArguments` and print every parameter

`TrainingArguments` is the **single configuration object** that controls the entire training loop — learning rate, batch size, epochs, precision, checkpointing, everything. We create it here with the **exact same values** used in the training notebook so you can see every parameter alongside its meaning and typical range. This is the complete reference you can come back to when tuning your own models.

In [ ]:
!pip install -q accelerate

In [ ]:
training_config = dict(
    output_dir="outputs/phayathaibert-intent",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=1,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    seed=42,
    logging_steps=25,
    warmup_ratio=0.0,
    report_to="none",
)

print("=== TrainingArguments we pass to the Trainer ===\n")
for k, v in training_config.items():
    print(f"  {k:<32} = {v}")

print("\n\n=== Complete Hyperparameter Reference ===")
params = [
    ("learning_rate",              training_config["learning_rate"],              "Step size for weight updates",        "1e-5 to 5e-5 for BERT"),
    ("num_train_epochs",           training_config["num_train_epochs"],           "Passes through training data",        "2-5 for fine-tuning"),
    ("per_device_train_batch_size",training_config["per_device_train_batch_size"],"Examples per GPU per step",           "8-32"),
    ("gradient_accumulation_steps",training_config["gradient_accumulation_steps"],"Virtual batch multiplier",            "1-8"),
    ("weight_decay",               training_config["weight_decay"],               "L2 regularization strength",         "0.0-0.1"),
    ("fp16",                       training_config["fp16"],                       "Half-precision training",             "True on GPU"),
    ("seed",                       training_config["seed"],                       "Reproducibility seed",                "Any integer"),
    ("warmup_ratio",               training_config["warmup_ratio"],               "LR warmup fraction of total steps",  "0.0-0.1"),
    ("logging_steps",              training_config["logging_steps"],              "Log training loss every N steps",     "10-100"),
    ("save_total_limit",           training_config["save_total_limit"],           "Max checkpoints to keep on disk",     "1-5"),
]

print(f"\n{'Parameter':<32} {'Value':>10} {'Meaning':<38} {'Typical Range'}")
print("-" * 110)
for name, val, meaning, range_ in params:
    print(f"{name:<32} {str(val):>10} {meaning:<38} {range_}")

print("\nIn the training notebook, these are passed as: TrainingArguments(**training_config)")

### Why we compute steps per epoch for different batch sizes

Students often ask: "If I change the batch size, what happens to the number of training steps?" This table answers that concretely. With `dataset_size=12` and `batch_size=8`, you get just 1 step per epoch. With `batch_size=2` and `accumulation=1`, you get 6 steps per epoch — more frequent weight updates but each one noisier. Understanding this interaction is key to tuning training speed vs quality.

In [ ]:
dataset_size = 12  # our small dataset

print("=== How epochs, batch size, and accumulation interact ===")
for bs in [2, 4, 8]:
    for accum in [1, 2, 4]:
        eff_bs = bs * accum
        steps_per_epoch = max(1, dataset_size // eff_bs)
        total_steps = steps_per_epoch * 3  # 3 epochs
        print(f"  batch_size={bs}, accum={accum} → effective_batch={eff_bs:>3}, "
              f"steps/epoch={steps_per_epoch:>3}, total_steps(3 epochs)={total_steps:>4}")

---

## Module 8: Safetensors

### What is safetensors?

A file format for storing neural network weights (tensors), created by Hugging Face as a **safe, fast replacement** for PyTorch's pickle-based `.bin` files.

After training, `trainer.save_model()` produces `model.safetensors` (~1.1 GB for PhayaThaiBERT).

### Why not just use `pytorch_model.bin`?

| Feature | `pytorch_model.bin` | `model.safetensors` |
|---------|-------------------|--------------------|
| Format | Python pickle | Custom binary (header + raw data) |
| **Security** | **Dangerous** — pickle can execute arbitrary code when loaded | **Safe** — no code execution, only tensor data |
| Loading speed | Slower (deserialize Python objects) | **Faster** (memory-mapped, zero-copy) |
| File size | Baseline | Same or slightly smaller |
| Framework support | PyTorch only | PyTorch, TensorFlow, JAX, ONNX Runtime, MLServer |

The security issue is real: a malicious `.bin` file downloaded from the internet could run **arbitrary code** on your machine when you call `torch.load()`. Safetensors is a flat map of tensor names to raw bytes — no executable code possible.

### File structure

```
[8 bytes: header length] [JSON header: tensor names, shapes, dtypes, byte offsets] [raw tensor data...]
```

The header is a JSON string listing every tensor with its name, data type (float16/float32), shape, and byte offset. The data section is contiguous raw bytes — no compression, no Python objects.

This makes it **memory-mappable**: the OS can load parts of the file on demand without reading the whole thing into RAM.

### Why we create, save, and inspect a safetensors file ourselves

Instead of just describing the format, we **build one from scratch**: create some dummy tensors, save them as safetensors, then read back the raw binary header to see the JSON metadata. This demystifies the file — it is not magic; it is just a structured binary with a JSON header listing tensor names, shapes, data types, and byte offsets, followed by raw float data. We also load it back and verify the shapes match, proving the round-trip works.

In [ ]:
import tempfile
from safetensors.torch import save_file, load_file

demo_tensors = {
    "layer.0.weight": torch.randn(768, 768),
    "layer.0.bias": torch.randn(768),
    "classifier.weight": torch.randn(3, 768),
    "classifier.bias": torch.randn(3),
}

with tempfile.NamedTemporaryFile(suffix=".safetensors", delete=False) as f:
    demo_path = f.name

save_file(demo_tensors, demo_path)

file_size = os.path.getsize(demo_path)
print(f"Saved demo safetensors: {file_size:,} bytes ({file_size/1024:.1f} KB)")

with open(demo_path, "rb") as f:
    header_len = struct.unpack("<Q", f.read(8))[0]
    header_json = json.loads(f.read(header_len))

print(f"\nHeader length: {header_len} bytes")
print(f"\n=== Tensor metadata in header ===")
for name, meta in header_json.items():
    if name == "__metadata__":
        continue
    print(f"  {name:<25} dtype={meta['dtype']:<10} shape={meta['data_offsets']}")

loaded = load_file(demo_path)
print(f"\n=== Loaded tensors ===")
for name, tensor in loaded.items():
    print(f"  {name:<25} shape={str(list(tensor.shape)):<15} dtype={tensor.dtype}")

os.unlink(demo_path)

### Complete list of artifacts saved after training

When `trainer.save_model()` and `tokenizer.save_pretrained()` are called, these files are written:

| File | Approx Size | Purpose | Who needs it |
|------|-------------|---------|-------------|
| `model.safetensors` | ~1.1 GB | All learned weights | MLServer, ONNX export |
| `config.json` | ~1 KB | Architecture definition (layers, hidden size, num_labels) | Everything |
| `tokenizer.json` | ~17 MB | Full vocabulary + merge rules | Tokenization at inference |
| `tokenizer_config.json` | ~0.5 KB | Tokenizer settings and special tokens | Tokenization at inference |
| `special_tokens_map.json` | ~1 KB | Special token definitions | Tokenization at inference |
| `label2id.json` | ~0.3 KB | Intent name to number mapping | Post-processing predictions |
| `run_meta.json` | ~0.3 KB | Reproducibility metadata (seed, split ratios) | Documentation |
| `training_args.bin` | ~5 KB | Trainer configuration snapshot | Reproducibility |

### How safetensors connects to deployment

| Serving Runtime | Format | Notes |
|----------------|--------|-------|
| **MLServer (Seldon)** | safetensors | Loads directly via `transformers.pipeline()` — **no conversion** |
| **OpenVINO Model Server** | ONNX | Requires conversion via `torch.onnx.export()` |
| **vLLM** | safetensors | Loads natively — but only for **generative** models (not classifiers) |

### Why we list the saved model artifacts

After training, the `outputs/phayathaibert-intent/final/` directory contains everything needed to reload and deploy the model. This cell lists every file with its actual size so you can see that `model.safetensors` is ~1.1 GB (the weights), `tokenizer.json` is ~17 MB (the vocabulary), and everything else is tiny metadata. If the directory doesn't exist (training hasn't run yet), we show the expected files so you know what to expect.

In [ ]:
model_dir = Path("outputs/phayathaibert-intent/final")

if model_dir.is_dir():
    print(f"=== Saved model artifacts in {model_dir} ===")
    for p in sorted(model_dir.iterdir()):
        if p.is_file():
            size = p.stat().st_size
            if size > 1e6:
                print(f"  {p.name:<30} {size/1e6:>10.1f} MB")
            else:
                print(f"  {p.name:<30} {size:>10,} bytes")
else:
    print(f"Model directory not found at {model_dir}.")
    print("Run the training notebook first to generate these files.")
    print("\nExpected files:")
    expected = [
        ("model.safetensors", "~1.1 GB"),
        ("config.json", "~1 KB"),
        ("tokenizer.json", "~17 MB"),
        ("tokenizer_config.json", "~0.5 KB"),
        ("special_tokens_map.json", "~1 KB"),
        ("label2id.json", "~0.3 KB"),
        ("run_meta.json", "~0.3 KB"),
    ]
    for name, size in expected:
        print(f"  {name:<30} {size}")

---

## Recap: Key Takeaways

| Concept | One-line summary |
|---------|------------------|
| **Model selection** | Match architecture (encoder/decoder), language, and size to your GPU |
| **Tokenizer** | Converts text to numbers — must match the pre-trained model exactly |
| **Full fine-tuning** | Update all weights — works for small models (< 1B params) |
| **LoRA** | Low-rank adapters — train 0.1-1% of params, freeze the rest |
| **QLoRA** | LoRA + 4-bit quantization — fits large models on small GPUs |
| **BitsAndBytes** | The library that implements 4-bit/8-bit quantization |
| **PEFT** | Umbrella term for parameter-efficient fine-tuning methods |
| **SFT** | Supervised fine-tuning — learning from labeled examples |
| **Learning rate** | Step size — `2e-5` is standard for BERT fine-tuning |
| **Loss function** | Cross-entropy: `-log(P(correct))` — penalizes confident wrong answers |
| **F1 score** | Harmonic mean of precision and recall — balanced metric |
| **Epochs** | Passes through the dataset — 2-5 for fine-tuning |
| **FP16** | Half-precision training — 2x speed, half memory |
| **Safetensors** | Safe, fast model weight format — no arbitrary code execution risk |

## What's Next?

Now that you understand every concept, open **`intent_classification_phayathaibert.ipynb`** and run the full training pipeline. You will see all of these concepts in action:

1. Load dataset and build label maps
2. Tokenize with `AutoTokenizer` (Module 2)
3. Load `AutoModelForSequenceClassification` (Module 1)
4. Configure `TrainingArguments` with learning rate, epochs, batch size, FP16 (Modules 4, 7)
5. Train with `Trainer` — watch the loss decrease (Module 5)
6. Evaluate with F1 and classification report (Module 6)
7. Save model as safetensors (Module 8)
8. Run inference on new Thai messages

For **Phase 2** (Qwen decoder), you will apply the QLoRA + BitsAndBytes + SFT concepts from Module 3.